In [11]:
import os
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)
from pymongo import MongoClient
from datetime import datetime

# 1. Verbindung zur MongoDB (Deine lokale Instanz auf dem Fedora/AlmaLinux)
client = MongoClient("mongodb://localhost:27017/")
db = client.blog # Wir legen es in die Auswanderungs-DB
collection = db.blogstats

# 2. GA4 Setup
# Pfad zu deiner heruntergeladenen JSON-Key Datei
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/home/skramer/Privat/Google_api_schlüssel/original-seeker-397717-496bfecad74a.json"
property_id = "514124337" 

def run_report(property_id):
    client_ga = BetaAnalyticsDataClient()

    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[Dimension(name="date")],
        metrics=[Metric(name="activeUsers"), Metric(name="sessions")],
        date_ranges=[DateRange(start_date="2026-03-01", end_date="today")],
    )
    
    response = client_ga.run_report(request)

    # Daten für MongoDB aufbereiten
    for row in response.rows:
        data_point = {
            "datum": datetime.strptime(row.dimension_values[0].value, "%Y%m%d"),
            "aktive_nutzer": int(row.metric_values[0].value),
            "sitzungen": int(row.metric_values[1].value),
            "blog": "sven-essen.de"
        }
        
        # Update oder Insert (Upset), damit wir keine Duplikate haben
        collection.update_one(
            {"datum": data_point["datum"], "blog": data_point["blog"]},
            {"$set": data_point},
            upsert=True
        )

run_report(property_id)
print("Blog-Statistiken erfolgreich in MongoDB synchronisiert!")

Blog-Statistiken erfolgreich in MongoDB synchronisiert!
